# Treasury Bond Futures CTD & Delivery Option Calculator

This Colab notebook estimates the **quality (bond-selection) delivery option** in a Treasury bond future. It follows the mechanism in *A Fat Cat in a Small Door* (Convexity Maven, 13 Oct 2026): the short chooses an eligible bond, and the CTD can change as yields move.

The notebook loads the official CME deliverable basket and conversion factors for the selected delivery month, imports current Treasury curve data and historical time series, ranks the CTD, analyzes CTD switching, and estimates the quality option value with a two-factor Monte Carlo model. CME basket and U.S. Treasury market data refresh each time the relevant cells are run. If the CME download is unavailable, the verified 8 Oct 2026 December 2026 basket is used as a labeled fallback.

**Model scope:** The option estimate depends on the basket, quote quality, yield-volatility assumptions, and delivery date. It models the quality option only. It omits wild-card, timing, end-of-month, repo specialness, delivery-month carry, cheapest financing, and delivery constraints. Treasury par yields are a public curve proxy, not individual cash-bond quotes; upload CUSIP-level quotes for precise CTD work.


Resources: [Source PDF](./Convexity-Maven-Fat-Cat.pdf) · [Open this notebook in Colab](https://colab.research.google.com/github/glxtraa/ConvexityCTD/blob/main/Treasury_Bond_Futures_CTD_Delivery_Option_Calculator.ipynb).

## 1. Model and interpretation

For each eligible bond, the model calculates its dirty price `P_i` and conversion-adjusted delivery cost `P_i / CF_i`. The CTD minimizes that cost. The short's modeled quality-option payoff compares the future delivery cost of today's CTD with the cheapest simulated delivery cost across the basket.

Yield scenarios use correlated normal level and slope changes; every bond is repriced at the simulated yield. The expected payoff is discounted to today. The yield process is a zero-drift approximation, not a calibrated arbitrage-free Hull-White model. Rolling realized volatility from Treasury yields is shown as an empirical baseline; use market-implied volatility when available and treat this model value as a sensitivity estimate rather than an exchange quote.

The December 2026 basket is refreshed from CME's conversion-factor workbook. A cached copy dated 8 October 2026 is included so the notebook remains usable if CME is temporarily unreachable. The current basket has 59 issues per that CME file.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import date, datetime
from IPython.display import display

np.random.seed(42)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
import requests


## 2. Contract, official basket, and market data

The CME basket updater uses the selected delivery month to select conversion factors from the latest CME workbook. If the workbook is unreachable for December 2026, the embedded 59-issue basket dated 8 October 2026 is used. The table displayed above is the contract's official eligible set, not the five highlighted examples in the source PDF.

Market data come from the U.S. Treasury's daily par-yield curve, with FRED's public CSV graph endpoint used to build the daily time series. The curve-based security yield is interpolated from the 10-, 20-, and 30-year par yields and is explicitly a proxy. For more accurate CTD analysis, upload timestamped individual CUSIP yields or prices below.


In [ ]:
# Contract / valuation inputs
valuation_date = pd.Timestamp.today().date()
delivery_date = date(2026, 12, 1)  # change to the contract delivery month/date
futures_price = 102.25             # enter current futures settlement/quote per 100 par
risk_free_discount_rate_pct = 4.0
number_of_scenarios = 30000
level_slope_correlation = -0.20
level_vol_bp_override = None       # optionally enter risk-neutral level sigma over the horizon
slope_vol_bp_override = None       # optionally enter risk-neutral slope sigma over the horizon

CME_PAGE = "https://www.cmegroup.com/trading/interest-rates/treasury-conversion-factors.html"
CME_CACHED_AS_OF = "2026-10-08"
CME_CACHED_ZB_DEC2026 = [{"cusip":"912810QU5","coupon_pct":3.125,"issue_date":"2012-02-15","maturity":"2042-02-15","conversion_factor":0.7182},{"cusip":"912810TF5","coupon_pct":2.375,"issue_date":"2022-02-28","maturity":"2042-02-15","conversion_factor":0.6447},{"cusip":"912810TH1","coupon_pct":3.25,"issue_date":"2022-05-31","maturity":"2042-05-15","conversion_factor":0.7277},{"cusip":"912810QW1","coupon_pct":3,"issue_date":"2012-05-15","maturity":"2042-05-15","conversion_factor":0.7029},{"cusip":"912810TK4","coupon_pct":3.375,"issue_date":"2022-08-31","maturity":"2042-08-15","conversion_factor":0.7375},{"cusip":"912810QX9","coupon_pct":2.75,"issue_date":"2012-08-15","maturity":"2042-08-15","conversion_factor":0.675},{"cusip":"912810TM0","coupon_pct":4,"issue_date":"2022-11-30","maturity":"2042-11-15","conversion_factor":0.798},{"cusip":"912810QY7","coupon_pct":2.75,"issue_date":"2012-11-15","maturity":"2042-11-15","conversion_factor":0.6718},{"cusip":"912810TQ1","coupon_pct":3.875,"issue_date":"2023-02-28","maturity":"2043-02-15","conversion_factor":0.7834},{"cusip":"912810QZ4","coupon_pct":3.125,"issue_date":"2013-02-15","maturity":"2043-02-15","conversion_factor":0.7069},{"cusip":"912810TS7","coupon_pct":3.875,"issue_date":"2023-05-31","maturity":"2043-05-15","conversion_factor":0.7813},{"cusip":"912810RB6","coupon_pct":2.875,"issue_date":"2013-05-15","maturity":"2043-05-15","conversion_factor":0.6784},{"cusip":"912810TU2","coupon_pct":4.375,"issue_date":"2023-08-31","maturity":"2043-08-15","conversion_factor":0.8313},{"cusip":"912810RC4","coupon_pct":3.625,"issue_date":"2013-08-15","maturity":"2043-08-15","conversion_factor":0.7534},{"cusip":"912810TW8","coupon_pct":4.75,"issue_date":"2023-11-30","maturity":"2043-11-15","conversion_factor":0.869},{"cusip":"912810RD2","coupon_pct":3.75,"issue_date":"2013-11-15","maturity":"2043-11-15","conversion_factor":0.7642},{"cusip":"912810TZ1","coupon_pct":4.5,"issue_date":"2024-02-29","maturity":"2044-02-15","conversion_factor":0.8415},{"cusip":"912810RE0","coupon_pct":3.625,"issue_date":"2014-02-18","maturity":"2044-02-15","conversion_factor":0.7491},{"cusip":"912810UB2","coupon_pct":4.625,"issue_date":"2024-05-31","maturity":"2044-05-15","conversion_factor":0.8534},{"cusip":"912810RG5","coupon_pct":3.375,"issue_date":"2014-05-15","maturity":"2044-05-15","conversion_factor":0.7202},{"cusip":"912810UD8","coupon_pct":4.125,"issue_date":"2024-09-03","maturity":"2044-08-15","conversion_factor":0.7986},{"cusip":"912810RH3","coupon_pct":3.125,"issue_date":"2014-08-15","maturity":"2044-08-15","conversion_factor":0.6911},{"cusip":"912810UF3","coupon_pct":4.625,"issue_date":"2024-12-02","maturity":"2044-11-15","conversion_factor":0.851},{"cusip":"912810RJ9","coupon_pct":3,"issue_date":"2014-11-17","maturity":"2044-11-15","conversion_factor":0.675},{"cusip":"912810UJ5","coupon_pct":4.75,"issue_date":"2025-02-28","maturity":"2045-02-15","conversion_factor":0.8635},{"cusip":"912810RK6","coupon_pct":2.5,"issue_date":"2015-02-17","maturity":"2045-02-15","conversion_factor":0.6179},{"cusip":"912810UL0","coupon_pct":5,"issue_date":"2025-06-02","maturity":"2045-05-15","conversion_factor":0.8899},{"cusip":"912810RM2","coupon_pct":3,"issue_date":"2015-05-15","maturity":"2045-05-15","conversion_factor":0.6699},{"cusip":"912810UN6","coupon_pct":4.875,"issue_date":"2025-09-02","maturity":"2045-08-15","conversion_factor":0.8753},{"cusip":"912810RN0","coupon_pct":2.875,"issue_date":"2015-08-17","maturity":"2045-08-15","conversion_factor":0.6536},{"cusip":"912810UQ9","coupon_pct":4.625,"issue_date":"2025-12-01","maturity":"2045-11-15","conversion_factor":0.8464},{"cusip":"912810RP5","coupon_pct":3,"issue_date":"2015-11-16","maturity":"2045-11-15","conversion_factor":0.665},{"cusip":"912810UT3","coupon_pct":4.625,"issue_date":"2026-03-02","maturity":"2046-02-15","conversion_factor":0.8454},{"cusip":"912810RQ3","coupon_pct":2.5,"issue_date":"2016-02-16","maturity":"2046-02-15","conversion_factor":0.6064},{"cusip":"912810UV8","coupon_pct":5,"issue_date":"2026-06-01","maturity":"2046-05-15","conversion_factor":0.8867},{"cusip":"912810RS9","coupon_pct":2.5,"issue_date":"2016-05-16","maturity":"2046-05-15","conversion_factor":0.6036},{"cusip":"912810UX4","coupon_pct":5.125,"issue_date":"2026-08-31","maturity":"2046-08-15","conversion_factor":0.9002},{"cusip":"912810RT7","coupon_pct":2.25,"issue_date":"2016-08-15","maturity":"2046-08-15","conversion_factor":0.5723},{"cusip":"912810RU4","coupon_pct":2.875,"issue_date":"2016-11-15","maturity":"2046-11-15","conversion_factor":0.6412},{"cusip":"912810RV2","coupon_pct":3,"issue_date":"2017-02-15","maturity":"2047-02-15","conversion_factor":0.6533},{"cusip":"912810RX8","coupon_pct":3,"issue_date":"2017-05-15","maturity":"2047-05-15","conversion_factor":0.651},{"cusip":"912810RY6","coupon_pct":2.75,"issue_date":"2017-08-15","maturity":"2047-08-15","conversion_factor":0.6195},{"cusip":"912810RZ3","coupon_pct":2.75,"issue_date":"2017-11-15","maturity":"2047-11-15","conversion_factor":0.6171},{"cusip":"912810SA7","coupon_pct":3,"issue_date":"2018-02-15","maturity":"2048-02-15","conversion_factor":0.6445},{"cusip":"912810SC3","coupon_pct":3.125,"issue_date":"2018-05-15","maturity":"2048-05-15","conversion_factor":0.6572},{"cusip":"912810SD1","coupon_pct":3,"issue_date":"2018-08-15","maturity":"2048-08-15","conversion_factor":0.6403},{"cusip":"912810SE9","coupon_pct":3.375,"issue_date":"2018-11-15","maturity":"2048-11-15","conversion_factor":0.6834},{"cusip":"912810SF6","coupon_pct":3,"issue_date":"2019-02-15","maturity":"2049-02-15","conversion_factor":0.6362},{"cusip":"912810SH2","coupon_pct":2.875,"issue_date":"2019-05-15","maturity":"2049-05-15","conversion_factor":0.6189},{"cusip":"912810SJ8","coupon_pct":2.25,"issue_date":"2019-08-15","maturity":"2049-08-15","conversion_factor":0.5403},{"cusip":"912810SK5","coupon_pct":2.375,"issue_date":"2019-11-15","maturity":"2049-11-15","conversion_factor":0.5532},{"cusip":"912810SL3","coupon_pct":2,"issue_date":"2020-02-18","maturity":"2050-02-15","conversion_factor":0.5045},{"cusip":"912810SN9","coupon_pct":1.25,"issue_date":"2020-05-15","maturity":"2050-05-15","conversion_factor":0.4086},{"cusip":"912810SP4","coupon_pct":1.375,"issue_date":"2020-08-17","maturity":"2050-08-15","conversion_factor":0.4213},{"cusip":"912810SS8","coupon_pct":1.625,"issue_date":"2020-11-16","maturity":"2050-11-15","conversion_factor":0.4499},{"cusip":"912810SU3","coupon_pct":1.875,"issue_date":"2021-02-16","maturity":"2051-02-15","conversion_factor":0.4789},{"cusip":"912810SX7","coupon_pct":2.375,"issue_date":"2021-05-17","maturity":"2051-05-15","conversion_factor":0.5399},{"cusip":"912810SZ2","coupon_pct":2,"issue_date":"2021-08-16","maturity":"2051-08-15","conversion_factor":0.49},{"cusip":"912810TB4","coupon_pct":1.875,"issue_date":"2021-11-15","maturity":"2051-11-15","conversion_factor":0.4716}]

def load_cme_basket(delivery_date):
    """Fetch the latest CME workbook and extract the ZB conversion-factor column."""
    target_month = pd.Timestamp(delivery_date).to_period("M").to_timestamp()
    try:
        import re, requests
        from io import BytesIO
        from urllib.parse import urljoin
        page = requests.get(CME_PAGE, timeout=30)
        page.raise_for_status()
        match = re.search(r"""href=["']([^"']*TCF\.xlsx[^"']*)["']""", page.text, flags=re.I)
        if not match:
            raise RuntimeError("The CME conversion-factor workbook link was not found")
        file_url = urljoin(CME_PAGE, match.group(1).replace("&amp;", "&"))
        response = requests.get(file_url, timeout=60)
        response.raise_for_status()
        raw = pd.read_excel(BytesIO(response.content), sheet_name="Conversion Factors", header=None)
        section = next(i for i in range(len(raw)) if str(raw.iat[i, 0]).strip() == "U.S. TREASURY BOND FUTURES CONTRACT")
        header = None
        target_col = None
        for i in range(section + 1, min(section + 10, len(raw))):
            for j, value in enumerate(raw.iloc[i].tolist()):
                if isinstance(value, (datetime, pd.Timestamp)) and pd.Timestamp(value).to_period("M").to_timestamp() == target_month:
                    header, target_col = i, j
                    break
            if header is not None:
                break
        if header is None:
            raise ValueError(f"CME workbook has no ZB conversion factors for {target_month:%b %Y}")
        records = []
        for i in range(header + 1, len(raw)):
            vals = raw.iloc[i].tolist()
            if i > section + 1 and isinstance(vals[0], str) and "FUTURES CONTRACT" in vals[0]:
                break
            cusip = vals[5] if len(vals) > 5 else None
            cf = vals[target_col] if target_col < len(vals) else None
            if isinstance(cusip, str) and cusip.strip().isdigit() and len(cusip.strip()) == 9 and isinstance(cf, (int, float)):
                records.append({"cusip": cusip.strip(), "coupon_pct": float(vals[2]),
                                "issue_date": pd.Timestamp(vals[3]).strftime("%Y-%m-%d"),
                                "maturity": pd.Timestamp(vals[4]).strftime("%Y-%m-%d"),
                                "conversion_factor": float(cf)})
        if not records:
            raise ValueError("No eligible ZB records were found in the CME workbook")
        print(f"Basket refreshed from CME: {len(records)} issues; workbook URL {file_url}")
        return pd.DataFrame(records), "CME live workbook"
    except Exception as exc:
        if target_month == pd.Timestamp("2026-12-01"):
            print(f"CME refresh unavailable ({exc}); using cached official basket dated {CME_CACHED_AS_OF}.")
            return pd.DataFrame(CME_CACHED_ZB_DEC2026), f"CME cached {CME_CACHED_AS_OF}"
        raise RuntimeError("Could not refresh the CME basket. Use a supported delivery month and check internet access.") from exc

basket, basket_source = load_cme_basket(delivery_date)
display(basket.head())
print(f"{len(basket)} eligible issues | Basket source: {basket_source}")


## 3. Refresh Treasury market data and time series

Run this cell to pull daily Treasury yields (no API key required), plot the 10-, 20-, and 30-year time series, and estimate rolling 60-session level and slope volatility. The rolling realized volatility is a historical reference input, not an implied volatility quote. You can override the calculated horizon volatility in the input cell above.


In [ ]:
# Download daily public Treasury constant-maturity yields through FRED's CSV graph endpoint.
fred_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10,DGS20,DGS30&cosd=2000-01-01"
market_data = pd.read_csv(fred_url, parse_dates=["observation_date"])
market_data = market_data.rename(columns={"observation_date":"date"}).set_index("date")
for col in ["DGS10", "DGS20", "DGS30"]:
    market_data[col] = pd.to_numeric(market_data[col], errors="coerce")
market_data = market_data.dropna(subset=["DGS10", "DGS20", "DGS30"])
if market_data.empty:
    raise ValueError("No Treasury yield rows downloaded. Check the public data connection.")
market_data["slope_30y_10y"] = market_data["DGS30"] - market_data["DGS10"]
market_data["level_vol_60d_ann_bp"] = market_data["DGS20"].diff().rolling(60).std() * np.sqrt(252) * 100
market_data["slope_vol_60d_ann_bp"] = market_data["slope_30y_10y"].diff().rolling(60).std() * np.sqrt(252) * 100

fig = go.Figure()
for col, label in [("DGS10", "10-year"), ("DGS20", "20-year"), ("DGS30", "30-year")]:
    fig.add_trace(go.Scatter(x=market_data.index, y=market_data[col], name=label, mode="lines"))
fig.update_layout(title="U.S. Treasury Constant-Maturity Yields", xaxis_title="Date",
                  yaxis_title="Yield (%)", hovermode="x unified", template="plotly_white")
fig.show()

vol_fig = make_subplots(specs=[[{"secondary_y":True}]])
vol_fig.add_trace(go.Scatter(x=market_data.index, y=market_data["level_vol_60d_ann_bp"],
                             name="20-year level volatility", line={"color":"#175a9c"}), secondary_y=False)
vol_fig.add_trace(go.Scatter(x=market_data.index, y=market_data["slope_vol_60d_ann_bp"],
                             name="30y-10y slope volatility", line={"color":"#ca6a1b"}), secondary_y=True)
vol_fig.update_layout(title="Rolling 60-session realized yield volatility (annualized)",
                      xaxis_title="Date", hovermode="x unified", template="plotly_white")
vol_fig.update_yaxes(title_text="Level volatility (bp/year)", secondary_y=False)
vol_fig.update_yaxes(title_text="Slope volatility (bp/year)", secondary_y=True)
vol_fig.show()

latest = market_data.dropna(subset=["DGS10", "DGS20", "DGS30"]).iloc[-1]
market_as_of = market_data.index[-1].date()
tenor_nodes = np.array([10.0, 20.0, 30.0])
par_yields = np.array([latest["DGS10"], latest["DGS20"], latest["DGS30"]], dtype=float)
years_to_delivery = max((pd.Timestamp(delivery_date) - pd.Timestamp(valuation_date)).days / 365.25, 1/365.25)
daily_level_vol = float(latest["level_vol_60d_ann_bp"]) if pd.notna(latest["level_vol_60d_ann_bp"]) else 50.0
daily_slope_vol = float(latest["slope_vol_60d_ann_bp"]) if pd.notna(latest["slope_vol_60d_ann_bp"]) else 25.0
level_vol_bp = level_vol_bp_override if level_vol_bp_override is not None else daily_level_vol * np.sqrt(years_to_delivery)
slope_vol_bp = slope_vol_bp_override if slope_vol_bp_override is not None else daily_slope_vol * np.sqrt(years_to_delivery)
print(f"Latest Treasury data: {market_as_of}; 10y/20y/30y par yields = {par_yields.tolist()}%")
print(f"Horizon vol inputs: level={level_vol_bp:.1f} bp, slope={slope_vol_bp:.1f} bp ({years_to_delivery:.2f} years)")


## 4. Optional CUSIP-level market quotes

Upload a CSV with at least `cusip,yield_pct` for same-time bond yields. The notebook joins by CUSIP and uses those yields where present; remaining issues use the Treasury par-curve proxy. A CSV template is available in the project folder. Keep the quote timestamp and source with the file for auditability.


In [ ]:
# Optional: set True and upload a CSV with columns cusip,yield_pct.
use_uploaded_cusip_quotes = False
if use_uploaded_cusip_quotes:
    from google.colab import files
    uploaded = files.upload()
    quote_file = next(iter(uploaded))
    quotes = pd.read_csv(quote_file)
    if not {"cusip", "yield_pct"}.issubset(quotes.columns):
        raise ValueError("Quote CSV must contain cusip and yield_pct columns")
    quotes["cusip"] = quotes["cusip"].astype(str).str.strip()
    quotes["yield_pct"] = pd.to_numeric(quotes["yield_pct"], errors="raise")
    basket = basket.merge(quotes[["cusip", "yield_pct"]], on="cusip", how="left")
    print(f"CUSIP yields loaded for {basket['yield_pct'].notna().sum()} of {len(basket)} eligible bonds")
else:
    basket["yield_pct"] = np.nan

years_to_maturity = (pd.to_datetime(basket["maturity"]) - pd.Timestamp(valuation_date)).dt.days.to_numpy() / 365.25
basket["curve_yield_pct"] = np.interp(years_to_maturity, tenor_nodes, par_yields)
basket["yield_pct"] = basket["yield_pct"].fillna(basket["curve_yield_pct"])
basket["yield_source"] = np.where(basket["curve_yield_pct"].notna(), "Treasury par-curve proxy", "Missing")
if use_uploaded_cusip_quotes:
    basket.loc[basket["cusip"].isin(quotes.loc[quotes["yield_pct"].notna(), "cusip"]), "yield_source"] = "Uploaded CUSIP yield"
display(basket[["cusip", "coupon_pct", "maturity", "conversion_factor", "yield_pct", "yield_source"]])


## 5. Shared calculation engine and current CTD

The next code cell embeds the same `ctd_model.py` source used by the Vercel API. This makes the notebook run its calculations locally in Colab; it does not call the website to calculate. When updating the model, copy the Vercel engine into this cell so both interfaces keep the same pricing, CTD ranking, scenario, discounting, and option payoff rules.


In [ ]:
"""Shared CTD and quality delivery-option calculations for Vercel and Colab."""

from __future__ import annotations

import calendar
from datetime import date, datetime
from typing import Any

import numpy as np


def _as_date(value: Any) -> date:
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    return date.fromisoformat(str(value)[:10])


def _shift_month(d: date, months: int) -> date:
    index = d.year * 12 + d.month - 1 + months
    year, month = divmod(index, 12)
    month += 1
    return date(year, month, min(d.day, calendar.monthrange(year, month)[1]))


def _coupon_dates(maturity: date, settlement: date) -> list[date]:
    dates = [maturity]
    while dates[-1] > settlement:
        dates.append(_shift_month(dates[-1], -6))
    return list(reversed(dates))


def bond_price(coupon_pct: float, maturity: Any, settlement: Any, yield_pct: float,
               face: float = 100.0) -> tuple[float, float, float]:
    """Return clean price, dirty price, and accrued interest per 100 par."""
    maturity, settlement = _as_date(maturity), _as_date(settlement)
    if maturity <= settlement:
        raise ValueError("Bond maturity must be after settlement")
    dates = _coupon_dates(maturity, settlement)
    previous, following = dates[0], dates[1]
    period_days = (following - previous).days
    accrued = face * coupon_pct / 100.0 / 2.0 * (settlement - previous).days / period_days
    first_fraction = (following - settlement).days / period_days
    y = float(yield_pct) / 100.0
    if 1.0 + y / 2.0 <= 0:
        raise ValueError("Yield is outside the supported range")
    coupon = face * coupon_pct / 100.0 / 2.0
    dirty = sum(
        (coupon + (face if pay_date == maturity else 0.0))
        / (1.0 + y / 2.0) ** (2.0 * (first_fraction + j))
        for j, pay_date in enumerate(dates[1:])
    )
    return dirty - accrued, dirty, accrued


def _dirty_price_vector(coupon_pct: float, maturity: Any, settlement: Any,
                        yields_pct: np.ndarray, face: float = 100.0) -> np.ndarray:
    maturity, settlement = _as_date(maturity), _as_date(settlement)
    dates = _coupon_dates(maturity, settlement)
    previous, following = dates[0], dates[1]
    period_days = (following - previous).days
    accrued = face * coupon_pct / 100.0 / 2.0 * (settlement - previous).days / period_days
    first_fraction = (following - settlement).days / period_days
    periods = first_fraction + np.arange(len(dates[1:]), dtype=float)
    cashflows = np.full(len(periods), face * coupon_pct / 100.0 / 2.0)
    cashflows[-1] += face
    y = np.asarray(yields_pct, dtype=float) / 100.0
    dirty = np.sum(cashflows[None, :] / (1.0 + y[:, None] / 2.0) ** (2.0 * periods[None, :]), axis=1)
    return dirty


def calculate(payload: dict[str, Any]) -> dict[str, Any]:
    """Calculate current CTD and a two-factor quality-option Monte Carlo estimate."""
    basket = payload.get("basket") or []
    if not basket:
        raise ValueError("Basket must contain at least one eligible bond")
    valuation_date = _as_date(payload["valuation_date"])
    delivery_date = _as_date(payload["delivery_date"])
    horizon = (delivery_date - valuation_date).days / 365.0
    if horizon <= 0:
        raise ValueError("Delivery date must be after valuation date")
    futures_price = float(payload.get("futures_price", 102.25))
    discount_rate_pct = float(payload.get("discount_rate_pct", 4.0))
    level_vol_bp = float(payload.get("level_vol_bp", 45.0))
    slope_vol_bp = float(payload.get("slope_vol_bp", 20.0))
    correlation = float(payload.get("correlation", -0.20))
    n_scenarios = int(payload.get("n_scenarios", 30000))
    seed = int(payload.get("seed", 42))
    if not -1 <= correlation <= 1:
        raise ValueError("Correlation must be between -1 and 1")
    if n_scenarios < 100 or n_scenarios > 200000:
        raise ValueError("Scenario count must be between 100 and 200,000")

    prepared = []
    for bond in basket:
        record = {
            "cusip": str(bond["cusip"]),
            "coupon_pct": float(bond["coupon_pct"]),
            "maturity": str(bond["maturity"])[:10],
            "conversion_factor": float(bond["conversion_factor"]),
            "yield_pct": float(bond["yield_pct"]),
            "yield_source": str(bond.get("yield_source", "Provided")),
        }
        if record["conversion_factor"] <= 0:
            raise ValueError(f"Invalid conversion factor for {record['cusip']}")
        clean, dirty, accrued = bond_price(record["coupon_pct"], record["maturity"], valuation_date, record["yield_pct"])
        record.update({
            "clean_price": clean,
            "dirty_price": dirty,
            "accrued": accrued,
            "adjusted_delivery_cost": dirty / record["conversion_factor"],
            "implied_futures_price": clean / record["conversion_factor"],
            "net_basis_points": (clean - futures_price * record["conversion_factor"]) * 100.0,
        })
        prepared.append(record)

    prepared.sort(key=lambda r: r["adjusted_delivery_cost"])
    current_ctd = prepared[0]
    # Deterministic parallel-shift analysis at valuation: apply the same yield
    # shock to every bond, reprice each issue, and find the nearest level where
    # another issue's conversion-adjusted cost crosses today's CTD.
    scenario_shocks = np.arange(-200.0, 201.0, 10.0)
    threshold_shocks = np.arange(-500.0, 501.0, 1.0)
    all_shocks = np.unique(np.concatenate([scenario_shocks, threshold_shocks]))
    scenario_costs = np.empty((len(all_shocks), len(prepared)), dtype=float)
    for i, bond in enumerate(prepared):
        shifted_yields = bond["yield_pct"] + all_shocks / 100.0
        scenario_costs[:, i] = _dirty_price_vector(
            bond["coupon_pct"], bond["maturity"], valuation_date, shifted_yields
        ) / bond["conversion_factor"]
    shock_index = {float(shock): i for i, shock in enumerate(all_shocks)}
    scenario_rows = []
    for shock in scenario_shocks:
        row = scenario_costs[shock_index[float(shock)]]
        scenario_rows.append({
            "parallel_shock_bp": float(shock),
            "ctd_cusip": prepared[int(np.argmin(row))]["cusip"],
            "ctd_adjusted_cost": float(np.min(row)),
        })
    ref_index = next(i for i, r in enumerate(prepared) if r["cusip"] == current_ctd["cusip"])
    thresholds = []
    current_cost_curve = scenario_costs[:, ref_index]
    for i, bond in enumerate(prepared):
        if i == ref_index:
            continue
        gap = scenario_costs[:, i] - current_cost_curve
        crossing = np.flatnonzero(gap[:-1] * gap[1:] <= 0.0)
        roots = []
        for j in crossing:
            x0, x1 = all_shocks[j], all_shocks[j + 1]
            y0, y1 = gap[j], gap[j + 1]
            root = x0 if y1 == y0 else x0 - y0 * (x1 - x0) / (y1 - y0)
            roots.append(float(root))
        thresholds.append({
            "cusip": bond["cusip"],
            "current_cost_gap": float(bond["adjusted_delivery_cost"] - current_ctd["adjusted_delivery_cost"]),
            "break_even_parallel_bp": min(roots, key=abs) if roots else None,
        })
    thresholds.sort(key=lambda r: (abs(r["break_even_parallel_bp"]) if r["break_even_parallel_bp"] is not None else float("inf"), r["current_cost_gap"]))
    switch_curves = []
    curve_indices = [ref_index] + [
        next(i for i, r in enumerate(prepared) if r["cusip"] == row["cusip"])
        for row in sorted(prepared, key=lambda r: r["adjusted_delivery_cost"])
        if row["cusip"] != current_ctd["cusip"]
    ][:7]
    for i in curve_indices:
        switch_curves.append({
            "cusip": prepared[i]["cusip"],
            "is_current_ctd": i == ref_index,
            "points": [
                {"parallel_shock_bp": float(shock),
                 "cost_gap": float(scenario_costs[shock_index[float(shock)], i] - scenario_costs[shock_index[float(shock)], ref_index])}
                for shock in scenario_shocks
            ],
        })
    delivery_years = np.array([(_as_date(r["maturity"]) - delivery_date).days / 365.25 for r in prepared])
    base_yields = np.array([r["yield_pct"] for r in prepared], dtype=float)
    cfs = np.array([r["conversion_factor"] for r in prepared], dtype=float)
    coupons = np.array([r["coupon_pct"] for r in prepared], dtype=float)
    rng = np.random.default_rng(seed)
    z = rng.multivariate_normal([[0.0, 0.0]][0], [[1.0, correlation], [correlation, 1.0]], size=n_scenarios)
    level = z[:, 0] * level_vol_bp / 10000.0
    slope = z[:, 1] * slope_vol_bp / 10000.0
    adjusted = np.empty((n_scenarios, len(prepared)), dtype=float)
    for i, bond in enumerate(prepared):
        scenario_yields = (base_yields[i] / 100.0 + level + slope * (delivery_years[i] - 20.0) / 10.0) * 100.0
        adjusted[:, i] = _dirty_price_vector(coupons[i], bond["maturity"], delivery_date, scenario_yields) / cfs[i]
    delivery_ctd = adjusted.argmin(axis=1)
    reference_cost = adjusted[:, ref_index]
    cheapest_cost = adjusted[np.arange(n_scenarios), delivery_ctd]
    payoff = np.maximum(reference_cost - cheapest_cost, 0.0)
    discount = float(np.exp(-discount_rate_pct / 100.0 * horizon))
    pv = payoff * discount
    counts = np.bincount(delivery_ctd, minlength=len(prepared))
    probabilities = counts / n_scenarios
    bin_counts, bin_edges = np.histogram(payoff, bins=50)

    return {
        "model_version": "ctd-quality-option-v1",
        "basket_size": len(prepared),
        "valuation_date": valuation_date.isoformat(),
        "delivery_date": delivery_date.isoformat(),
        "current_ctd": current_ctd["cusip"],
        "current_ctd_adjusted_dirty_cost": current_ctd["adjusted_delivery_cost"],
        "current_rows": prepared,
        "parallel_shock_scenarios": scenario_rows,
        "ctd_switch_thresholds": thresholds,
        "ctd_switch_curves": switch_curves,
        "summary": {
            "horizon_years": horizon,
            "reference_cusip": current_ctd["cusip"],
            "option_pv_per_100_par": float(pv.mean()),
            "option_pv_per_1000_par": float(pv.mean() * 10.0),
            "undiscounted_expected_payoff": float(payoff.mean()),
            "mc_standard_error": float(pv.std(ddof=1) / np.sqrt(n_scenarios)),
            "probability_current_ctd_is_not_cheapest": float(np.mean(delivery_ctd != ref_index)),
            "probability_option_payoff_is_positive": float(np.mean(payoff > 0.0)),
            "discount_factor": discount,
            "scenario_count": n_scenarios,
        },
        "ctd_probabilities": [
            {"cusip": prepared[i]["cusip"], "probability": float(probabilities[i])}
            for i in range(len(prepared))
        ],
        "payoff_histogram": {"counts": bin_counts.tolist(), "edges": bin_edges.tolist()},
    }


In [ ]:
def run_shared_model(level_vol=None, slope_vol=None, scenarios=None):
    payload = {
        "valuation_date": str(valuation_date),
        "delivery_date": str(delivery_date),
        "futures_price": float(futures_price),
        "discount_rate_pct": float(risk_free_discount_rate_pct),
        "level_vol_bp": float(level_vol_bp if level_vol is None else level_vol),
        "slope_vol_bp": float(slope_vol_bp if slope_vol is None else slope_vol),
        "correlation": float(level_slope_correlation),
        "n_scenarios": int(number_of_scenarios if scenarios is None else scenarios),
        "basket": basket[["cusip", "coupon_pct", "maturity", "conversion_factor", "yield_pct", "yield_source"]].to_dict(orient="records"),
    }
    return calculate(payload)

shared_result = run_shared_model()
summary = shared_result["summary"]
current = pd.DataFrame(shared_result["current_rows"])
ctd = current.iloc[0]
print(f"Current CTD: {shared_result['current_ctd']} | adjusted dirty cost: {shared_result['current_ctd_adjusted_dirty_cost']:.4f}")
print(f"Quality-option PV: ${summary['option_pv_per_100_par']:.5f} per $100 par | MC standard error: ${summary['mc_standard_error']:.5f}")
display(current[["cusip", "coupon_pct", "maturity", "conversion_factor", "yield_pct", "yield_source", "clean_price", "dirty_price", "adjusted_delivery_cost", "net_basis_points"]])


## 6. Yield scenarios and delivery-option estimate

The embedded model applies correlated level and slope yield changes, reprices every eligible bond, and estimates the quality option at delivery. It also applies deterministic parallel yield shocks at today’s valuation date to show the rate move required for an alternative bond to tie the current CTD. It runs locally in this notebook.


In [ ]:
ctd_table = pd.DataFrame(shared_result["ctd_probabilities"])
payoff_hist = shared_result["payoff_histogram"]


## 7. CTD-switch, break-even, and payoff diagnostics


In [ ]:
display(ctd_table.sort_values("probability", ascending=False))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Cheapest-to-deliver at delivery", "Undiscounted quality-option payoff"))
fig.add_trace(go.Bar(x=ctd_table["cusip"], y=ctd_table["probability"]*100,
                     name="CTD probability", marker_color="#245b8f"), row=1, col=1)
edges = payoff_hist["edges"]
counts = payoff_hist["counts"]
centers = [(edges[i] + edges[i+1])/2 for i in range(len(counts))]
fig.add_trace(go.Bar(x=centers, y=counts, name="Payoff distribution",
                     marker_color="#ca6a1b"), row=1, col=2)
fig.update_xaxes(title_text="CUSIP", tickangle=45, row=1, col=1)
fig.update_yaxes(title_text="Probability (%)", row=1, col=1)
fig.update_xaxes(title_text="Payoff per $100 par", row=1, col=2)
fig.update_yaxes(title_text="Scenarios", row=1, col=2)
fig.update_layout(template="plotly_white", height=450, showlegend=False,
                  title="Delivery-option scenario diagnostics")
fig.show()

# Deterministic break-even rate shifts: a positive shock means higher yields.
switch_thresholds = pd.DataFrame(shared_result["ctd_switch_thresholds"])
switch_thresholds["break_even_parallel_bp"] = switch_thresholds["break_even_parallel_bp"].map(
    lambda x: "No tie within ±500 bp" if pd.isna(x) else round(x, 1)
)
display(switch_thresholds.head(12))

stress_ctds = pd.DataFrame(shared_result["parallel_shock_scenarios"])
display(stress_ctds[stress_ctds["parallel_shock_bp"].isin([-200, -100, -50, 0, 50, 100, 200])])

scenario_fig = go.Figure()
for curve in shared_result["ctd_switch_curves"]:
    points = pd.DataFrame(curve["points"])
    label = curve["cusip"] + (" · current CTD" if curve["is_current_ctd"] else "")
    scenario_fig.add_trace(go.Scatter(
        x=points["parallel_shock_bp"], y=points["cost_gap"], mode="lines", name=label
    ))
scenario_fig.add_hline(y=0, line_color="#536b80", line_width=1)
scenario_fig.update_layout(
    template="plotly_white", height=500,
    title="Parallel yield shock required for a CTD switch",
    xaxis_title="Parallel yield shock (bp; positive = higher yields)",
    yaxis_title="Adjusted cost gap vs current CTD ($ per $100 par)",
    hovermode="x unified",
)
scenario_fig.show()


## 8. Volatility sensitivity

Each sensitivity point calls the embedded calculation engine. The engine source matches the Vercel calculator so basket valuation, scenario generation, and option formulas follow the same rules in both interfaces.


In [ ]:
vol_rows = []
for scale in [0.50, 0.75, 1.00, 1.25, 1.50]:
    result = run_shared_model(level_vol_bp*scale, slope_vol_bp*scale,
                              max(4000, number_of_scenarios//8))
    s = result["summary"]
    vol_rows.append({"volatility_scale":scale, "level_vol_bp":level_vol_bp*scale,
                     "slope_vol_bp":slope_vol_bp*scale,
                     "option_pv_per_100_par":s["option_pv_per_100_par"],
                     "ctd_switch_probability":s["probability_current_ctd_is_not_cheapest"]})
vol_sensitivity = pd.DataFrame(vol_rows)
display(vol_sensitivity)
fig = go.Figure()
fig.add_trace(go.Scatter(x=vol_sensitivity["volatility_scale"], y=vol_sensitivity["option_pv_per_100_par"],
                         mode="lines+markers", name="Option PV", line={"color":"#175a9c"}))
fig.add_trace(go.Scatter(x=vol_sensitivity["volatility_scale"], y=vol_sensitivity["ctd_switch_probability"]*100,
                         mode="lines+markers", name="CTD switch probability (%)", yaxis="y2",
                         line={"color":"#ca6a1b"}))
fig.update_layout(title="Quality-option sensitivity to rate volatility", xaxis_title="Volatility scale",
                  yaxis_title="PV per $100 par", yaxis2={"title":"CTD switch probability (%)", "overlaying":"y", "side":"right"},
                  hovermode="x unified", template="plotly_white")
fig.show()


## 9. Practical use and source notes

1. The basket cell downloads CME's latest workbook, then selects the U.S. Treasury Bond Futures (ZB) conversion-factor column for the chosen delivery month. It falls back to the verified December 2026 basket cached from CME's 8 October 2026 workbook.
2. Treasury daily par-yield history is downloaded from FRED's public graph CSV endpoint (DGS10, DGS20, DGS30). The latest curve is linearly interpolated to each eligible bond's remaining maturity. This is a proxy yield; upload same-timestamp CUSIP-specific yields to improve CTD accuracy.
3. Enter a current futures quote in `futures_price`. The web calculator uses `ctd_model.py`; this stand-alone notebook embeds that same calculation engine and runs it locally. The public curve is not an executable cash or futures quote.
4. For option valuation, prefer risk-neutral implied volatility inputs. The rolling 60-session realized factor volatilities are empirical estimates and may differ materially from implied volatility.

The official basket workbook reports 59 eligible issues for December 2026 and was updated 8 October 2026. CME says conversion factors are specific to a delivery month and define invoice conversion. See [CME Treasury Conversion Factors](https://www.cmegroup.com/trading/interest-rates/treasury-conversion-factors.html), [CME conversion-factor explanation](https://www.cmegroup.com/articles/2024/calculating-us-treasury-futures-conversion-factors.html), and [U.S. Treasury daily yield feed](https://home.treasury.gov/treasury-daily-interest-rate-xml-feed). The source research, *Convexity Maven, “A Fat Cat in a Small Door,” pages 2–6 (13 October 2026)*, motivates CTD extension and delivery-option risk but does not provide a calibrated option premium.

This notebook is for research and education, not investment advice. Results can be materially wrong when the quote data, basket month, volatility inputs, or delivery assumptions are stale or incomplete.
